<a href="https://colab.research.google.com/github/buildwithdemis/machinelearning/blob/main/Winnipeg_Transit_On_Time_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Business problem:

Public transit reliability is an important factor in how citizens experience their daily commute. In Winnipeg, buses occasionally arrive either ahead of schedule, right on time, or later than planned. This unpredictability can cause inconvenience for riders, impact connections between routes, and reduce overall confidence in the transit system.

Our goal in this project is to explore whether we can use historical transit schedule and performance data to predict the punctuality of buses. By looking at patterns such as the route number, stop location, time of day, and day of week, we aim to forecast whether a bus is likely to be early, on time, or late.

# Type of ML problem:
   ## Classification (Early, On-time vs Late).
   * This is a classification problem in machine learning, since we are assigning each bus trip to one of three categories: Early, On-time, or Late.

The overall objective is to train and evaluate a predictive model within Azure Machine Learning, and then log and register the model using MLflow so that it can be managed and potentially deployed as part of a larger intelligent transit system.

# Goal:
   Train, evaluate, and register a model in Azure ML.

# Dataset Selection

we are using the Winnipeg Transit On-Time Performance dataset from the open data portal.
* source: https://data.winnipeg.ca/Transit/Recent-Transit-On-Time-Performance-Data/gp3k-am4u/about_data

* Date range: May 17,2025 to Aug. 19, 2025
* since the whole data size is larger than 600Mb, i just take the first 100k rows for the purpose of demo

# Load the data

In [26]:
!wget https://raw.githubusercontent.com/buildwithdemis/machinelearning/refs/heads/main/al/Transit_On-Time_Performance_100k.csv

--2025-09-01 19:45:30--  https://raw.githubusercontent.com/buildwithdemis/machinelearning/refs/heads/main/al/Transit_On-Time_Performance_100k.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13206334 (13M) [text/plain]
Saving to: ‘Transit_On-Time_Performance_100k.csv.1’

Transit_On-Time_Per 100%[===================>]  12.59M  --.-KB/s    in 0.1s    

2025-09-01 19:45:31 (114 MB/s) - ‘Transit_On-Time_Performance_100k.csv.1’ saved [13206334/13206334]



In [44]:
import pandas as pd
# Read the text file containing data using pandas
df = pd.read_csv('Transit_On-Time_Performance_100k.csv', delimiter=',')


print("number of rows: " , df.size)

# Because there are a lot of data, use head() to only print the first few rows
df.head(10)

number of rows:  900000


,Row ID,Stop Number,Route Number,Route Name,Route Destination,Day Type,Scheduled Time,Deviation,Location
0,1529721475,40198,D17,Talbot - Selkirk,Kildonan Place,Saturday,2025 Jul 05 08:28:45 AM,190,POINT (-97.0636837759858 49.8993976819722)
1,1529707518,31013,39,Inkster,Waterford Green,Saturday,2025 Jul 05 10:07:13 PM,-57,POINT (-97.1300872125569 49.9294967207071)
2,1529682770,10339,43,Watt - Logan,Gateway,Saturday,2025 Jul 05 11:58:02 AM,173,POINT (-97.1504360488864 49.9071647664343)
3,1529693051,30165,38,Mountain - Munroe,Kildonan Place,Saturday,2025 Jul 05 04:31:19 PM,-130,POINT (-97.1361307804554 49.9216514650652)
4,1529696396,60450,70,Roblin,Unicity,Saturday,2025 Jul 05 08:32:43 PM,48,POINT (-97.2036614122655 49.8747203214952)
5,1529612457,60064,91,St. Norbert,St. Norbert,Saturday,2025 Jul 05 07:13:00 AM,-192,POINT (-97.1565841949361 49.7896341634893)
6,1529696447,60556,70,Roblin,Unicity,Saturday,2025 Jul 05 08:51:48 PM,64,POINT (-97.2924548225466 49.8594937363161)
7,1529696398,60444,70,Roblin,Unicity,Saturday,2025 Jul 05 08:33:35 PM,69,POINT (-97.2054612101309 49.8715999527232)
8,1529696520,60461,70,Roblin,Polo Park,Saturday,2025 Jul 05 09:41:56 PM,166,POINT (-97.2136300871861 49.8663381400167)
9,1529612543,60026,91,St. Norbert,St. Norbert,Saturday,2025 Jul 05 07:52:42 AM,-49,POINT (-97.1569697482449 49.7794757683298)


In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   Row ID             100000 non-null  int64 
 1   Stop Number        100000 non-null  int64 
 2   Route Number       100000 non-null  object
 3   Route Name         100000 non-null  object
 4   Route Destination  100000 non-null  object
 5   Day Type           100000 non-null  object
 6   Scheduled Time     100000 non-null  object
 7   Deviation          100000 non-null  object
 8   Location           100000 non-null  object
dtypes: int64(2), object(7)
memory usage: 6.9+ MB


## Description of each column

* Stop Number (int64): Numeric ID of the bus stop where the observation was recorded. Each stop number uniquely identifies a physical bus stop in Winnipeg.
* Route Number (object): The official route number of the bus (e.g., 16, 21, 60, D13,F5). Routes with higher traffic or longer distances may show different on-time patterns.

* Route Name (object): Human-readable name of the bus route (e.g., “GRANT EXPRESS”). Often correlated with route number but provides more descriptive context.
* Route Destination (object): The end destination of the route for that trip (e.g., “Downtown”, “Kildonan Place”). Useful to distinguish between directions of travel on the same route.
* Day Type (object): Indicates the type of service day (e.g., “Weekday”, “Saturday”, “Sunday/Holiday”). Bus schedules and traffic patterns differ significantly by day type.

* Scheduled Time (object): The scheduled departure or arrival time at the bus stop (e.g., “08:45:00”).Needs to be converted into a time-based feature (hour of day, morning/evening, peak vs off-peak).
* Location (object): The geographic description of the stop location (e.g., “Portage & Main”).Can be used for spatial analysis or grouped by high-traffic areas.

* Deviation (object): The difference between actual arrival time and scheduled time. Usually expressed in minutes (negative = early, zero = on time, positive = late).

This is the target variable, which we will classify into categories (Early, On-time, Late).

Label = Late if offset > 60 sec

Label = On-time if offset between 0-60 sec

Label = EARLY if offset ≤ 0 sec

## Apply One-Hot Encoding

The Day Type column is categorical (Weekday, Saturday, Sunday/Holiday), so we’ll need to convert it into numeric values for use in machine learning.
since i am planning to use linear models (e.g., Logistic Regression, Neural Networks), one-hot encoding is better.


In [46]:
# Generate dummies separately
daytype_dummies = pd.get_dummies(df["Day Type"], prefix="DayType")

# Drop duplicates if they exist before concatenation
for col in daytype_dummies.columns:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Concatenate back
df = pd.concat([df, daytype_dummies], axis=1)

# Convert bool → int
df[daytype_dummies.columns] = df[daytype_dummies.columns].astype(int)

In [42]:
df.head()

,Row ID,Stop Number,Route Number,Route Name,Route Destination,Day Type,Scheduled Time,Deviation,Location,DayType_Saturday,DayType_Sunday,DayType_Weekday
0,1529721475,40198,D17,Talbot - Selkirk,Kildonan Place,Saturday,2025 Jul 05 08:28:45 AM,190,POINT (-97.0636837759858 49.8993976819722),1,0,0
1,1529707518,31013,39,Inkster,Waterford Green,Saturday,2025 Jul 05 10:07:13 PM,-57,POINT (-97.1300872125569 49.9294967207071),1,0,0
2,1529682770,10339,43,Watt - Logan,Gateway,Saturday,2025 Jul 05 11:58:02 AM,173,POINT (-97.1504360488864 49.9071647664343),1,0,0
3,1529693051,30165,38,Mountain - Munroe,Kildonan Place,Saturday,2025 Jul 05 04:31:19 PM,-130,POINT (-97.1361307804554 49.9216514650652),1,0,0
4,1529696396,60450,70,Roblin,Unicity,Saturday,2025 Jul 05 08:32:43 PM,48,POINT (-97.2036614122655 49.8747203214952),1,0,0


Now we have additional three columns:
* DayType_Saturday
* DayType_Sunday
* DayType_Weekday

## Finding Missing Data
Do we have a complete dataset? we use isnull() to report the columns that have "empty" cells:

In [47]:
missing_data = df.isnull().sum().to_frame()
# Rename column holding the sums
missing_data = missing_data.rename(columns={0:'Empty Cells'})
missing_data

,Empty Cells
Row ID,0
Stop Number,0
Route Number,0
Route Name,0
Route Destination,0
Day Type,0
Scheduled Time,0
Deviation,0
Location,0
DayType_Saturday,0


After running the check, we observe that there are no missing values in the dataset. Therefore, we can proceed with our analysis without needing to drop any rows or perform data imputation.

## Label Engineering (Deviation → Status)

The Deviation column in the dataset represents the difference (in seconds) between a bus’s actual arrival time and its scheduled arrival time. While this is a continuous numeric value, `the business problem we want to solve is not about predicting the exact number of seconds a bus will be early or late. Instead, the goal is to predict whether a bus will be early, on time, or late.`

To achieve this, we transform Deviation into a categorical column called Status using defined thresholds:

	* EARLY: Bus arrives more than 1 minutes before scheduled time (Deviation < -60).
	* ON_TIME: Bus arrives within ±2 minutes of schedule (-60 ≤ Deviation ≤ 60).
	* LATE: Bus arrives more than 2 minutes after scheduled time (Deviation > 60).

This transformation aligns the dataset with a classification problem, making it suitable for training predictive models in Azure ML.

Additionally, the Status column is encoded into a numeric Status_Label column so that machine learning algorithms can process it effectively.

In [51]:
def categorize_deviation(value):
    if value < -60:       # more than 1 min early
        return "EARLY"
    elif value > 60:      # more than 1 min late
        return "LATE"
    else:
        return "ON_TIME"

df["Status"] = df["Deviation"].str.replace(',', '').astype(int).apply(categorize_deviation)

# Task
Explain the error in the selected code, fix it if possible, and incorporate the changes into the existing code. Otherwise, diagnose the error.

## Encode status column

### Subtask:
Encode the 'Status' column into a numeric 'Status_Label' column.


**Reasoning**:
Encode the 'Status' column into a numeric 'Status_Label' column by mapping the string values to integers and then converting the new column to integer type.



In [52]:
status_mapping = {"EARLY": 0, "ON_TIME": 1, "LATE": 2}
df["Status_Label"] = df["Status"].map(status_mapping)
df["Status_Label"] = df["Status_Label"].astype(int)

## Select features

### Subtask:
Select the features to be used for training the model.


**Reasoning**:
Create the features list, the label variable, and the feature and label dataframes according to the instructions.



In [53]:
features = ['Stop Number', 'Route Number', 'DayType_Saturday', 'DayType_Sunday', 'DayType_Weekday']
label = 'Status_Label'
X = df[features]
y = df[label]

## Split data

### Subtask:
Split the data into training and testing sets.


**Reasoning**:
Split the data into training and testing sets using train_test_split.



In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Train model

### Subtask:
Train a classification model to predict the 'Status_Label'.


**Reasoning**:
Import the necessary library for the chosen classification model and train the model using the training data.



In [55]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

ValueError: could not convert string to float: 'D10'

**Reasoning**:
The traceback indicates a ValueError because the 'Route Number' column contains string values ('D10') that cannot be converted to floats, which is required by the RandomForestClassifier. I need to encode the 'Route Number' column to a numerical representation before training the model. I will use one-hot encoding for this categorical feature.



In [56]:
# Apply one-hot encoding to 'Route Number'
route_dummies = pd.get_dummies(X_train["Route Number"], prefix="Route")

# Drop the original 'Route Number' column and concatenate the new dummy columns
X_train = X_train.drop(columns=["Route Number"])
X_train = pd.concat([X_train, route_dummies], axis=1)

# Convert bool → int
X_train[route_dummies.columns] = X_train[route_dummies.columns].astype(int)

# Do the same for the test set to ensure consistency
route_dummies_test = pd.get_dummies(X_test["Route Number"], prefix="Route")
X_test = X_test.drop(columns=["Route Number"])
X_test = pd.concat([X_test, route_dummies_test], axis=1)

# Convert bool → int
X_test[route_dummies_test.columns] = X_test[route_dummies_test.columns].astype(int)

# Now train the model again with the numerically encoded features
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Evaluate model

### Subtask:
Evaluate the performance of the trained model.


**Reasoning**:
Import the necessary function for model evaluation and generate predictions on the test set.



In [57]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- Route_223
- Route_334
- Route_444
- Route_641
- Route_642
- ...


**Reasoning**:
The error indicates a mismatch in feature names between the training and testing sets after one-hot encoding. This is because the test set may not contain all categories present in the training set, leading to missing columns in the one-hot encoded test set. To fix this, we need to ensure both training and test sets have the same one-hot encoded columns by using the `columns` attribute of the one-hot encoded training set when processing the test set. Then, generate predictions and the classification report.



In [58]:
# Apply one-hot encoding to 'Route Number' for both train and test sets
# Ensure consistent columns by using the columns from the training set after encoding
route_dummies_train = pd.get_dummies(X_train["Route Number"], prefix="Route")
route_dummies_test = pd.get_dummies(X_test["Route Number"], prefix="Route")

# Reindex the test set dummy columns to match the training set dummy columns
route_dummies_test = route_dummies_test.reindex(columns=route_dummies_train.columns, fill_value=0)


# Drop the original 'Route Number' column and concatenate the new dummy columns for both train and test sets
X_train = X_train.drop(columns=["Route Number"])
X_train = pd.concat([X_train, route_dummies_train], axis=1)

X_test = X_test.drop(columns=["Route Number"])
X_test = pd.concat([X_test, route_dummies_test], axis=1)

# Convert bool → int for both train and test sets
X_train[route_dummies_train.columns] = X_train[route_dummies_train.columns].astype(int)
X_test[route_dummies_test.columns] = X_test[route_dummies_test.columns].astype(int)


# Re-train the model with the corrected X_train
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions on the corrected X_test
y_pred = model.predict(X_test)

# Generate and print the classification report
print(classification_report(y_test, y_pred))

KeyError: 'Route Number'

**Reasoning**:
The `KeyError: 'Route Number'` indicates that the 'Route Number' column was already dropped from `X_train` in a previous attempt within the same code block. We need to ensure that the one-hot encoding is applied to the original `X_train` and `X_test` before dropping the column and concatenating. The previous code block attempted to perform the one-hot encoding and concatenation for both train and test within the same block, which led to the column being dropped twice. I will restructure the code to perform the one-hot encoding and concatenation correctly for both train and test sets, then retrain the model and generate the classification report.



In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

# Apply one-hot encoding to 'Route Number' for train set
route_dummies_train = pd.get_dummies(X_train["Route Number"], prefix="Route")
X_train_processed = X_train.drop(columns=["Route Number"])
X_train_processed = pd.concat([X_train_processed, route_dummies_train], axis=1)
X_train_processed[route_dummies_train.columns] = X_train_processed[route_dummies_train.columns].astype(int)

# Apply one-hot encoding to 'Route Number' for test set and reindex to match train set columns
route_dummies_test = pd.get_dummies(X_test["Route Number"], prefix="Route")
route_dummies_test = route_dummies_test.reindex(columns=route_dummies_train.columns, fill_value=0)
X_test_processed = X_test.drop(columns=["Route Number"])
X_test_processed = pd.concat([X_test_processed, route_dummies_test], axis=1)
X_test_processed[route_dummies_test.columns] = X_test_processed[route_dummies_test.columns].astype(int)


# Train the model with the processed training data
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

# Make predictions on the processed test data
y_pred = model.predict(X_test_processed)

# Generate and print the classification report
print(classification_report(y_test, y_pred))

KeyError: 'Route Number'

**Reasoning**:
The error persists because `X_train` and `X_test` were modified in a previous cell, and the 'Route Number' column was dropped. The current code block is trying to access the 'Route Number' column from these already modified DataFrames. To fix this, I need to re-create `X_train` and `X_test` from the original `df` DataFrame before applying the one-hot encoding in this cell.



In [60]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd
from sklearn.model_selection import train_test_split

# Re-create X and y from the original df
features = ['Stop Number', 'Route Number', 'DayType_Saturday', 'DayType_Sunday', 'DayType_Weekday']
label = 'Status_Label'
X = df[features]
y = df[label]

# Re-split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply one-hot encoding to 'Route Number' for train set
route_dummies_train = pd.get_dummies(X_train["Route Number"], prefix="Route")
X_train_processed = X_train.drop(columns=["Route Number"])
X_train_processed = pd.concat([X_train_processed, route_dummies_train], axis=1)
X_train_processed[route_dummies_train.columns] = X_train_processed[route_dummies_train.columns].astype(int)


# Apply one-hot encoding to 'Route Number' for test set and reindex to match train set columns
route_dummies_test = pd.get_dummies(X_test["Route Number"], prefix="Route")
route_dummies_test = route_dummies_test.reindex(columns=route_dummies_train.columns, fill_value=0)
X_test_processed = X_test.drop(columns=["Route Number"])
X_test_processed = pd.concat([X_test_processed, route_dummies_test], axis=1)
X_test_processed[route_dummies_test.columns] = X_test_processed[route_dummies_test.columns].astype(int)


# Train the model with the processed training data
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

# Make predictions on the processed test data
y_pred = model.predict(X_test_processed)

# Generate and print the classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.72      0.67      9111
           1       0.45      0.42      0.44      6896
           2       0.49      0.39      0.44      3993

    accuracy                           0.55     20000
   macro avg       0.52      0.51      0.51     20000
weighted avg       0.54      0.55      0.54     20000



## Log and register model

### Subtask:
Log and register the model using MLflow in Azure ML.


**Reasoning**:
Import the necessary libraries and get the Azure ML workspace.



In [61]:
import mlflow
import mlflow.sklearn
from azureml.core import Workspace

ModuleNotFoundError: No module named 'mlflow'

**Reasoning**:
Install the missing mlflow library using pip.



In [62]:
!pip install mlflow azureml-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1

**Reasoning**:
Import the necessary libraries and get the Azure ML workspace.



In [63]:
from azureml.core import Workspace
import mlflow
import mlflow.sklearn
from sklearn.metrics import classification_report

# Get the Azure ML Workspace
try:
    ws = Workspace.from_config()
except Exception as e:
    print(f"Could not retrieve workspace from config: {e}")
    print("Please make sure you have configured your Azure ML workspace.")
    ws = None


Could not retrieve workspace from config: UserErrorException:
	Message: The workspace configuration file config.json, could not be found in /content or its parent directories. Please check whether the workspace configuration file exists, or provide the full path to the configuration file as an argument. You can download a configuration file for your workspace, via http://ml.azure.com and clicking on the name of your workspace in the right top.
	InnerException None
	ErrorResponse 
{
    "error": {
        "code": "UserError",
        "message": "The workspace configuration file config.json, could not be found in /content or its parent directories. Please check whether the workspace configuration file exists, or provide the full path to the configuration file as an argument. You can download a configuration file for your workspace, via http://ml.azure.com and clicking on the name of your workspace in the right top."
    }
}
Please make sure you have configured your Azure ML workspace.


## Log and register model

### Subtask:
Log and register the model using MLflow in Azure ML.


**Reasoning**:
Since the previous attempt to connect to the Azure ML workspace failed due to missing configuration, I will provide placeholder values for the workspace details to continue with the MLflow logging and registration steps as outlined in the subtask instructions. This will allow the code to run and demonstrate the MLflow functionality, even without a live connection.



In [64]:
# Provide placeholder values for Azure ML workspace details
# In a real scenario, replace these with your actual workspace details
# from a config.json file or by explicitly defining them.
subscription_id = 'YOUR_SUBSCRIPTION_ID'
resource_group = 'YOUR_RESOURCE_GROUP'
workspace_name = 'YOUR_WORKSPACE_NAME'

# Set the MLflow tracking URI to the Azure ML workspace URI
# Construct a placeholder tracking URI
azureml_tracking_uri = f'azureml://eastus.api.azureml.ms/mlflow/v1.0/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/{workspace_name}'
mlflow.set_tracking_uri(azureml_tracking_uri)

# Start an MLflow run
with mlflow.start_run() as run:
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

    # Log parameters
    mlflow.log_param("model_name", "RandomForestClassifier")
    mlflow.log_param("random_state", 42)

    # Make predictions on the test set to get metrics
    y_pred = model.predict(X_test_processed)

    # Evaluate the model and get the classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Log metrics from the classification report
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                 mlflow.log_metric(f"{label}_{metric_name}", value)
        else:
             mlflow.log_metric(label, metrics)


    # Log the trained scikit-learn model
    model_name = "bus-punctuality-model"
    mlflow.sklearn.log_model(model, model_name)

    # Construct a placeholder model URI for registration
    model_uri = f"runs:/{run_id}/{model_name}"

    # Register the logged model in Azure ML Model Registry
    # This step will likely fail without a valid Azure ML connection,
    # but it demonstrates the intended workflow.
    try:
        registered_model = mlflow.register_model(
            model_uri=model_uri,
            name=model_name,
            tags={"task": "classification", "data": "winnipeg-transit"},
            description="Random Forest Classifier for bus punctuality prediction"
        )
        print(f"Model registered: {registered_model.name}, version: {registered_model.version}")
    except Exception as e:
        print(f"Failed to register model: {e}")

# The MLflow run is automatically ended when exiting the 'with' block
print("MLflow run finished.")


UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'azureml://eastus.api.azureml.ms/mlflow/v1.0/subscriptions/YOUR_SUBSCRIPTION_ID/resourceGroups/YOUR_RESOURCE_GROUP/providers/Microsoft.MachineLearningServices/workspaces/YOUR_WORKSPACE_NAME' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https', 'postgresql', 'mysql', 'sqlite', 'mssql']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.

**Reasoning**:
The traceback indicates that the MLflow tracking URI set in the previous step is not recognized as a valid URI for model registry functionality by the current MLflow configuration. The error message lists the supported URI schemes, and `azureml` is not among them for model registry. To address this and continue with the MLflow logging steps (excluding the model registration which requires a valid Azure ML connection setup), I will remove the model registration part and ensure the model logging still attempts to save the model locally within the run's artifacts. I will also explicitly set the tracking URI to a local file path to avoid the unsupported URI error during `mlflow.start_run`.



In [65]:
# Set the MLflow tracking URI to a local file path
mlflow.set_tracking_uri("file:///tmp/mlruns")

# Start an MLflow run
with mlflow.start_run() as run:
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

    # Log parameters
    mlflow.log_param("model_name", "RandomForestClassifier")
    mlflow.log_param("random_state", 42)

    # Make predictions on the test set to get metrics
    y_pred = model.predict(X_test_processed)

    # Evaluate the model and get the classification report
    report = classification_report(y_test, y_pred, output_dict=True)

    # Log metrics from the classification report
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, value in metrics.items():
                 mlflow.log_metric(f"{label}_{metric_name}", value)
        else:
             mlflow.log_metric(label, metrics)

    # Log the trained scikit-learn model
    model_name = "bus-punctuality-model"
    mlflow.sklearn.log_model(model, model_name)
    print(f"Model logged as artifact: {model_name}")

# The MLflow run is automatically ended when exiting the 'with' block
print("MLflow run finished.")


MLflow Run ID: 0c26dee1dd0e4b66bc75ad91473c1d61


2025/09/01 20:35:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/09/01 20:35:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model logged as artifact: bus-punctuality-model
MLflow run finished.


**Reasoning**:
The previous step successfully logged the model and metrics locally using MLflow. The subtask instructions also include registering the model in Azure ML. Although direct registration failed due to the tracking URI setup, the model was logged as an artifact. To fulfill the registration part of the subtask, I will now attempt to register the model using `mlflow.register_model` by referencing the run ID and artifact path where the model was logged. This step is still dependent on a proper Azure ML connection configuration, but it completes the sequence of actions outlined in the subtask.

